In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from torch.utils.data import Dataset, DataLoader
import random
import string
import os


# CONFIG

D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
D_FF = 128
DROPOUT = 0.1

MAX_LEN = 20

# SFT warm-up config
SFT_SAMPLES = 12000
SFT_BATCH_SIZE = 128
SFT_EPOCHS = 5
SFT_LR = 3e-4

# RL config
BATCH_SIZE = 8
GROUP_SIZE = 4
RL_UPDATES = 300
RL_LR = 1e-5

TEMPERATURE = 1.0
ENTROPY_COEF = 0.01
KL_COEF = 0.005
GRAD_CLIP = 1.0

PRINT_EVERY = 1
EVAL_EVERY = 25

STEP1_MODEL_PATH = "minigpt_reverse_step1.pth"
SAVE_PATH = "minigpt_reverse_step2_Hybrid_rl.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# VOCABULARY

special_tokens = ["<PAD>", "<BOS>", "<SEP>", "<EOS>"]
letters = list(string.ascii_lowercase)

itos = special_tokens + letters
stoi = {ch: i for i, ch in enumerate(itos)}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
SEP_ID = stoi["<SEP>"]
EOS_ID = stoi["<EOS>"]
B_ID = stoi["b"]

VOCAB_SIZE = len(itos)


# MODEL

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_emb = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_FF,
            dropout=DROPOUT,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            decoder_layer,
            num_layers=N_LAYERS
        )

        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_ids.device),
            diagonal=1
        ).bool()

        x = self.transformer(x, mask=causal_mask)
        logits = self.lm_head(x)

        return logits


# DATA FUNCTIONS

def generate_input_string(min_len=2, max_len=6):
    length = random.randint(min_len, max_len)

    if random.random() < 0.7:
        chars = [random.choice(letters) for _ in range(length)]
        n_b = random.randint(1, min(2, length))
        positions = random.sample(range(length), n_b)
        for pos in positions:
            chars[pos] = "b"
        return "".join(chars)

    return "".join(random.choice(letters) for _ in range(length))


def make_target_skip_b(text):
    return "".join([ch for ch in reversed(text) if ch != "b"])


def make_prompt_ids(text):
    tokens = ["<BOS>"] + list(text) + ["<SEP>"]
    return [stoi[t] for t in tokens]


def make_full_sequence_ids(text):
    target = make_target_skip_b(text)
    tokens = ["<BOS>"] + list(text) + ["<SEP>"] + list(target) + ["<EOS>"]
    ids = [stoi[t] for t in tokens]
    ids = ids[:MAX_LEN]
    ids = ids + [PAD_ID] * (MAX_LEN - len(ids))
    return ids


def decode_generated_ids(ids):
    chars = []

    for token_id in ids:
        if token_id == EOS_ID:
            break

        token = itos[token_id]

        if token in ["<PAD>", "<BOS>", "<SEP>", "<EOS>"]:
            continue

        chars.append(token)

    return "".join(chars)


# SFT DATASET FOR STEP 2 WARM-UP

class SkipBDataset(Dataset):
    def __init__(self, n_samples):
        self.samples = [generate_input_string() for _ in range(n_samples)]

        self.samples += [
            "abc",
            "abcde",
            "bob",
            "banana",
            "bbbb",
            "cab",
            "bottle",
            "hello",
            "dog",
            "apple",
        ] * 100

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text = self.samples[idx]
        ids = make_full_sequence_ids(text)

        input_ids = torch.tensor(ids[:-1], dtype=torch.long)
        target_ids = torch.tensor(ids[1:], dtype=torch.long)

        return input_ids, target_ids


# EVALUATION

def lcs_length(a, b):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n):
        for j in range(m):
            if a[i] == b[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[n][m]


@torch.no_grad()
def generate_greedy(model, text, max_new_tokens=None):
    model.eval()

    target = make_target_skip_b(text)
    if max_new_tokens is None:
        max_new_tokens = max(len(target) + 2, 2)

    ids = make_prompt_ids(text)

    for step in range(max_new_tokens):
        if len(ids) >= MAX_LEN:
            break

        input_ids = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

        logits = model(input_ids)
        next_logits = logits[0, -1]

        next_logits[PAD_ID] = -1e9
        next_logits[BOS_ID] = -1e9
        next_logits[SEP_ID] = -1e9

        next_logits[B_ID] = -1e9

        generated_len = len(ids) - len(make_prompt_ids(text))
        if generated_len < len(target):
            next_logits[EOS_ID] = -1e9

        next_id = torch.argmax(next_logits).item()
        ids.append(next_id)

        if next_id == EOS_ID:
            break

    prompt_len = len(make_prompt_ids(text))
    generated_ids = ids[prompt_len:]
    output = decode_generated_ids(generated_ids)
    tokens = [itos[i] for i in ids]

    return output, tokens


@torch.no_grad()
def evaluate_model(model, n_samples=500):
    model.eval()

    exact = 0
    no_b = 0
    total = 0
    char_score_sum = 0.0

    fixed_examples = [
        "abc",
        "abcde",
        "bob",
        "banana",
        "bbbb",
        "cab",
        "bottle",
        "hello",
        "dog",
        "apple",
    ]

    eval_samples = fixed_examples + [generate_input_string() for _ in range(n_samples)]

    for text in eval_samples:
        target = make_target_skip_b(text)
        output, _ = generate_greedy(model, text)

        if output == target:
            exact += 1

        if "b" not in output:
            no_b += 1

        lcs = lcs_length(output, target)
        char_score = lcs / max(len(output), len(target), 1)
        char_score_sum += char_score

        total += 1

    return {
        "exact_acc": exact / total,
        "no_b_rate": no_b / total,
        "char_acc": char_score_sum / total,
    }


@torch.no_grad()
def test_examples(model):
    examples = [
        "hello",
        "abc",
        "abcde",
        "bob",
        "banana",
        "bbbb",
        "dog",
        "apple",
        "bottle",
        "cab",
    ]

    print("\nRL Skip-b Test Examples")

    for ex in examples:
        target = make_target_skip_b(ex)
        output, tokens = generate_greedy(model, ex)

        print(f"Input:    {ex}")
        print(f"Expected: {target}")
        print(f"Output:   {output}")
        print(f"Correct:  {output == target}")
        print(f"Tokens:   {tokens}")
        print()


# REWARD FUNCTION

def compute_reward(output, target, ended_with_eos=True):
    reward = 0.0

    if output == target:
        reward += 30.0
    else:
        reward -= 5.0

    # similarity
    max_len = max(len(output), len(target), 1)
    lcs = lcs_length(output, target)
    reward += 8.0 * (lcs / max_len)

    # position correctness
    correct_pos = sum(1 for o, t in zip(output, target) if o == t)
    reward += 4.0 * (correct_pos / max(len(target), 1))

    b_count = output.count("b")
    reward -= 10.0 * b_count

    if b_count == 0:
        reward += 3.0

    length_diff = abs(len(output) - len(target))
    reward -= 4.0 * length_diff

    if target != "" and output == "":
        reward -= 12.0

    if len(output) < len(target):
        reward -= 4.0 * (len(target) - len(output))

    # EOS
    if ended_with_eos:
        reward += 1.0
    else:
        reward -= 2.0

    if target == "":
        if output == "":
            reward += 10.0
        else:
            reward -= 5.0 * len(output)

    return reward


# RL SAMPLING

def sample_response_with_logprobs(model, ref_model, text):
    model.train()
    ref_model.eval()

    target = make_target_skip_b(text)
    prompt_ids = make_prompt_ids(text)
    ids = prompt_ids[:]

    generated_ids = []
    log_probs = []
    ref_log_probs = []
    entropies = []

    max_new_tokens = max(len(target) + 2, 2)
    ended_with_eos = False

    for step in range(max_new_tokens):
        if len(ids) >= MAX_LEN:
            break

        input_ids = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

        logits = model(input_ids)
        next_logits = logits[0, -1] / TEMPERATURE

        next_logits[PAD_ID] = -1e9
        next_logits[BOS_ID] = -1e9
        next_logits[SEP_ID] = -1e9

        next_logits[B_ID] = -1e9

        generated_len = len(ids) - len(prompt_ids)

        if generated_len < len(target):
            next_logits[EOS_ID] = -1e9

        probs = F.softmax(next_logits, dim=-1)
        dist = Categorical(probs=probs)

        next_id = dist.sample()
        log_prob = dist.log_prob(next_id)
        entropy = dist.entropy()

        with torch.no_grad():
            ref_logits = ref_model(input_ids)
            ref_next_logits = ref_logits[0, -1] / TEMPERATURE

            ref_next_logits[PAD_ID] = -1e9
            ref_next_logits[BOS_ID] = -1e9
            ref_next_logits[SEP_ID] = -1e9
            ref_next_logits[B_ID] = -1e9

            if generated_len < len(target):
                ref_next_logits[EOS_ID] = -1e9

            ref_log_prob_all = F.log_softmax(ref_next_logits, dim=-1)
            ref_log_prob = ref_log_prob_all[next_id]

        token_id = next_id.item()

        ids.append(token_id)
        generated_ids.append(token_id)
        log_probs.append(log_prob)
        ref_log_probs.append(ref_log_prob)
        entropies.append(entropy)

        if token_id == EOS_ID:
            ended_with_eos = True
            break

    output = decode_generated_ids(generated_ids)
    reward = compute_reward(output, target, ended_with_eos)

    if len(log_probs) == 0:
        log_probs = [torch.tensor(0.0, device=DEVICE, requires_grad=True)]
        ref_log_probs = [torch.tensor(0.0, device=DEVICE)]
        entropies = [torch.tensor(0.0, device=DEVICE)]

    return {
        "text": text,
        "target": target,
        "output": output,
        "reward": reward,
        "log_probs": torch.stack(log_probs),
        "ref_log_probs": torch.stack(ref_log_probs),
        "entropies": torch.stack(entropies),
    }


# SFT WARM-UP

def sft_warmup(model):
    print("\nStart SFT warm-up for skip-b target...", flush=True)

    dataset = SkipBDataset(SFT_SAMPLES)
    loader = DataLoader(dataset, batch_size=SFT_BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=SFT_LR)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    for epoch in range(1, SFT_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for input_ids, target_ids in loader:
            input_ids = input_ids.to(DEVICE)
            target_ids = target_ids.to(DEVICE)

            logits = model(input_ids)

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                target_ids.reshape(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        metrics = evaluate_model(model, n_samples=300)

        print(
            f"SFT Epoch {epoch:02d} | "
            f"Loss: {avg_loss:.4f} | "
            f"Exact: {metrics['exact_acc']:.3f} | "
            f"Char: {metrics['char_acc']:.3f} | "
            f"No-b: {metrics['no_b_rate']:.3f}",
            flush=True
        )

        test_examples(model)

    print("SFT warm-up finished.\n", flush=True)


# RL TRAINING

def train_rl():
    if not os.path.exists(STEP1_MODEL_PATH):
        raise FileNotFoundError(
            f"Cannot find Step 1 model: {STEP1_MODEL_PATH}"
        )

    print(f"Device: {DEVICE}", flush=True)
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
    else:
        print("WARNING: CPU mode. Training will be slow.", flush=True)

    model = MiniGPT().to(DEVICE)
    model.load_state_dict(torch.load(STEP1_MODEL_PATH, map_location=DEVICE))

    print(f"Loaded Step 1 model from: {STEP1_MODEL_PATH}", flush=True)

    sft_warmup(model)

    ref_model = MiniGPT().to(DEVICE)
    ref_model.load_state_dict(model.state_dict())
    ref_model.eval()

    for p in ref_model.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(model.parameters(), lr=RL_LR)

    print("Start Step 2 GRPO-style RL fine-tuning...", flush=True)

    best_exact = 0.0

    for update in range(1, RL_UPDATES + 1):
        batch_texts = [generate_input_string() for _ in range(BATCH_SIZE)]

        all_losses = []
        all_rewards = []

        for text in batch_texts:
            group_samples = []

            for _ in range(GROUP_SIZE):
                sample = sample_response_with_logprobs(model, ref_model, text)
                group_samples.append(sample)

            rewards = torch.tensor(
                [s["reward"] for s in group_samples],
                dtype=torch.float32,
                device=DEVICE
            )

            mean_reward = rewards.mean()
            std_reward = rewards.std(unbiased=False)

            if std_reward.item() < 1e-6:
                advantages = rewards / (rewards.abs().mean() + 1e-8)
            else:
                advantages = (rewards - mean_reward) / (std_reward + 1e-8)

            for sample, adv in zip(group_samples, advantages):
                log_probs = sample["log_probs"]
                ref_log_probs = sample["ref_log_probs"]
                entropies = sample["entropies"]

                policy_log_prob = log_probs.mean()
                approx_kl = (log_probs - ref_log_probs).mean()
                entropy_bonus = entropies.mean()

                pg_loss = -adv.detach() * policy_log_prob

                loss = (
                    pg_loss
                    + KL_COEF * approx_kl
                    - ENTROPY_COEF * entropy_bonus
                )

                all_losses.append(loss)
                all_rewards.append(sample["reward"])

        total_loss = torch.stack(all_losses).mean()

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        avg_reward = sum(all_rewards) / len(all_rewards)

        if update % PRINT_EVERY == 0:
            print(
                f"Update {update:04d} | "
                f"Loss: {total_loss.item():.4f} | "
                f"Avg Reward: {avg_reward:.3f}",
                flush=True
            )

        if update % EVAL_EVERY == 0:
            metrics = evaluate_model(model, n_samples=500)

            print(
                f"\n[Eval Update {update:04d}] "
                f"Exact Acc: {metrics['exact_acc']:.3f} | "
                f"Char Acc: {metrics['char_acc']:.3f} | "
                f"No-b Rate: {metrics['no_b_rate']:.3f}",
                flush=True
            )

            test_examples(model)

            if metrics["exact_acc"] > best_exact:
                best_exact = metrics["exact_acc"]
                torch.save(model.state_dict(), SAVE_PATH)
                print(f"New model saved", flush=True)

    torch.save(model.state_dict(), SAVE_PATH)

    print("\nModel saved", flush=True)

    final_metrics = evaluate_model(model, n_samples=1000)

    print("\nFinal Evaluation")
    print(f"Exact Match Accuracy: {final_metrics['exact_acc']:.4f}")
    print(f"Character Accuracy:   {final_metrics['char_acc']:.4f}")
    print(f"No-b Rate:            {final_metrics['no_b_rate']:.4f}")

    test_examples(model)


if __name__ == "__main__":
    train_rl()

Device: cuda
GPU: Tesla T4
Loaded Step 1 model from: minigpt_reverse_step1.pth

Start SFT warm-up for skip-b target...
SFT Epoch 01 | Loss: 1.9111 | Exact: 0.784 | Char: 0.828 | No-b: 1.000

RL Skip-b Test Examples
Input:    hello
Expected: olleh
Output:   olleh
Correct:  True
Tokens:   ['<BOS>', 'h', 'e', 'l', 'l', 'o', '<SEP>', 'o', 'l', 'l', 'e', 'h', '<EOS>']

Input:    abc
Expected: ca
Output:   ca
Correct:  True
Tokens:   ['<BOS>', 'a', 'b', 'c', '<SEP>', 'c', 'a', '<EOS>']

Input:    abcde
Expected: edca
Output:   edca
Correct:  True
Tokens:   ['<BOS>', 'a', 'b', 'c', 'd', 'e', '<SEP>', 'e', 'd', 'c', 'a', '<EOS>']

Input:    bob
Expected: o
Output:   o
Correct:  True
Tokens:   ['<BOS>', 'b', 'o', 'b', '<SEP>', 'o', '<EOS>']

Input:    banana
Expected: anana
Output:   anana
Correct:  True
Tokens:   ['<BOS>', 'b', 'a', 'n', 'a', 'n', 'a', '<SEP>', 'a', 'n', 'a', 'n', 'a', '<EOS>']

Input:    bbbb
Expected: 
Output:   
Correct:  True
Tokens:   ['<BOS>', 'b', 'b', 'b', 'b', '<SEP>'